### Librerias

In [1]:
import pandas as pd
from google.cloud import bigquery
import os

/home/guy3hil/vertex_dev/lib/python3.8/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.8.20). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/home/guy3hil/vertex_dev/lib/python3.8/site-packages/google/auth/__init__.py:52: FutureWarning: You are using a Python version 3.8 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.8"), FutureWarning)
/home/guy3hil/vertex_dev/lib/python3.8/site-packages/google/oauth2/__init__.py:38: FutureWarning: You are using a Python version 3.8 past its end of life. Google will update google-auth with critic

### Datos

In [7]:
Datos = pd.read_csv('/home/guy3hil/Clase 28 - Proyecto Final/iris.csv') 
Datos.head(5)

,Id,SepalLength,SepalWidth,PetalLength,PetalWidth,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


### Crear Train y Test data

In [44]:
from sklearn.model_selection import train_test_split

Datos = pd.read_csv('/home/guy3hil/Clase 28 - Proyecto Final/iris.csv')

Train, Test = train_test_split(Datos, test_size=0.2, random_state=42)

columnas = ['SepalLength', 'SepalWidth', 'PetalLength', 'PetalWidth', 'Species']
Train = Train[columnas].reset_index(drop=True)
Test = Test[columnas].reset_index(drop=True)

Train.to_csv('/home/guy3hil/Clase 28 - Proyecto Final/data/iris-train.csv', index=False)
Test.to_csv('/home/guy3hil/Clase 28 - Proyecto Final/data/iris-test.csv', index=False)

Test.head(5)

,SepalLength,SepalWidth,PetalLength,PetalWidth,Species
0,6.1,2.8,4.7,1.2,Iris-versicolor
1,5.7,3.8,1.7,0.3,Iris-setosa
2,7.7,2.6,6.9,2.3,Iris-virginica
3,6.0,2.9,4.5,1.5,Iris-versicolor
4,6.8,2.8,4.8,1.4,Iris-versicolor


### Carga tablas a BigQuery

In [45]:
# Crear un cliente para interactuar con BigQuery
client = bigquery.Client(project="proyecto-fuente")

# Definir el ID del proyecto y el ID del dataset donde se cargarán los datos
project_id = "proyecto-fuente"  # Reemplazar con el ID de tu proyecto de Google Cloud
dataset_id = "BaseDatosIris"  # Reemplazar con el ID del dataset donde se almacenarán las tablas
folder_path = "/home/guy3hil/Clase 28 - Proyecto Final/data/"  # Ruta local de la carpeta que contiene los archivos CSV

# Referencia al dataset en BigQuery
dataset_ref = client.dataset(dataset_id)

# Iterar sobre los archivos dentro de la carpeta especificada
for file_name in os.listdir(folder_path):
    # Filtrar solo los archivos con extensión .csv
    if file_name.endswith(".csv"):
        # Crear un ID de tabla basado en el nombre del archivo (sin la extensión .csv)
        table_id = os.path.splitext(file_name)[0]
        table_ref = dataset_ref.table(table_id)  # Crear una referencia a la tabla de destino en BigQuery

        # Configurar el trabajo de carga (Load Job) para BigQuery
        job_config = bigquery.LoadJobConfig(
            source_format=bigquery.SourceFormat.CSV,  # El formato de origen es CSV
            #skip_leading_rows=1,  # Ignorar la primera fila del CSV (por ejemplo, los encabezados de las columnas)
            autodetect=True,  # Detectar automáticamente el esquema de las columnas del CSV
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE  # Overwrite existing table
        )

        # Construir la ruta completa del archivo CSV
        csv_file_path = os.path.join(folder_path, file_name)

        # Abrir el archivo CSV y cargarlo a BigQuery
        with open(csv_file_path, "rb") as source_file:
            # Iniciar el trabajo de carga a BigQuery desde el archivo CSV
            load_job = client.load_table_from_file(source_file, table_ref, job_config=job_config)

        # Imprimir el estado del trabajo antes de que comience
        print(f"Starting job {load_job.job_id} for {file_name}")
        
        # Esperar a que el trabajo de carga se complete
        load_job.result()

        # Imprimir el estado del trabajo una vez terminado
        print(f"Job {load_job.job_id} finished for {file_name}")

        # Obtener la tabla de destino para verificar cuántas filas fueron cargadas
        destination_table = client.get_table(table_ref)
        
        # Imprimir el número de filas que se han cargado en la tabla
        print(f"Loaded {destination_table.num_rows} rows into {table_ref}.")

Starting job aafe0708-d900-4e40-9469-9301db0bd33d for iris-train.csv
Job aafe0708-d900-4e40-9469-9301db0bd33d finished for iris-train.csv
Loaded 120 rows into proyecto-fuente.BaseDatosIris.iris-train.
Starting job fe464115-c0c4-426b-ac0b-9b77d4b17af5 for iris-test.csv
Job fe464115-c0c4-426b-ac0b-9b77d4b17af5 finished for iris-test.csv
Loaded 30 rows into proyecto-fuente.BaseDatosIris.iris-test.
